# Chapter 3 Lab — ReAct Loop + JSON-Schema Constrained Output

**Estimated time:** 25 minutes · **Read-and-run.** A mock LLM + mock search tool;
no API keys, no network.

In [ ]:
# A tiny knowledge base our 'search' tool can hit.
KB = {
  'nvidia nim': 'NVIDIA NIM is a set of inference microservices for deploying models.',
  'triton': 'Triton Inference Server serves models with dynamic batching and multiple backends.',
  'tensorrt-llm': 'TensorRT-LLM compiles optimized inference engines for LLMs on NVIDIA GPUs.'}
def search_tool(q):
    q=q.lower()
    hits=[v for k,v in KB.items() if k in q or any(t in q for t in k.split())]
    return hits[0] if hits else 'no result'

In [ ]:
# A deterministic 'policy' standing in for an LLM: Thought -> Action -> (Observation) -> Answer.
def mock_policy(question, scratch):
    if not scratch:
        return {'type':'action','tool':'search','input':question}
    obs = scratch[-1]['observation']
    return {'type':'final','answer':obs}

def react(question, max_steps=4):
    scratch=[]
    for step in range(max_steps):
        act = mock_policy(question, scratch)
        print(f'Thought {step}: need info' if act['type']=='action' else f'Thought {step}: have enough')
        if act['type']=='final':
            return act['answer']
        obs = search_tool(act['input'])
        print(f'  Action: search({act["input"]!r}) -> Observation: {obs[:60]}...')
        scratch.append({'action':act,'observation':obs})
    return 'gave up'
answer = react('What is NVIDIA NIM?')
print('\nFinal:', answer)

## Constrain the final answer to a JSON schema

In [ ]:
SCHEMA = {'answer': str, 'source': str, 'confidence': float}
def validate(obj, schema):
    errs=[]
    for key, typ in schema.items():
        if key not in obj: errs.append(f'missing {key}')
        elif not isinstance(obj[key], typ): errs.append(f'{key} must be {typ.__name__}')
    return errs

good = {'answer': answer, 'source':'KB', 'confidence':0.9}
bad  = {'answer': answer, 'confidence':'high'}  # wrong type + missing field
print('good ->', validate(good, SCHEMA) or 'VALID')
print('bad  ->', validate(bad, SCHEMA))

## Key takeaway
ReAct adds **tool access**, chain-of-thought adds **reasoning depth**, and schema-constrained
decoding fixes **output format**. Each targets a different failure mode — picking the wrong
one burns tokens without fixing the real problem.